# 4.8 Applications of Vector Spaces

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 4 — Vector Spaces**

---

### What you will learn

- How the **solution space** of a linear ODE forms a vector space.
- How to compute the **Wronskian** to test whether functions are linearly independent.
- The connection between the Wronskian and the **determinant** from Chapter 3.
- How to extend these ideas to **higher-order** differential equations.

### Prerequisites

- Vector spaces, subspaces, basis, and dimension (Notebooks 4.1–4.7).
- Determinants via cofactor expansion and elimination (Notebooks 3.1–3.2).
- The `det_cofactor` and `det_elimination` functions from `linalg_core.determinant`.

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.determinant import det_cofactor, det_elimination
from linalg_core import EPSILON
import math

print("Setup complete.")

---

## 2. Motivation

So far, vector spaces have contained *vectors* — columns of numbers. But the
theory is far more general. The set of all *functions* on an interval can form
a vector space (with pointwise addition and scalar multiplication), and the
**solution set of a homogeneous linear differential equation** is a *subspace*
of that function space.

This means that all the machinery we built — span, linear independence, basis,
dimension — applies directly to differential equations. The key computational
tool linking the two worlds is the **Wronskian**, which is nothing more than a
determinant evaluated on functions and their derivatives.

> **Differential equations have solution spaces that are vector spaces.**
> **The Wronskian tests independence.**

---

## 3. Build

We build up from a concrete ODE, through the Wronskian, to higher-order
equations, and finally connect everything back to the matrix determinant.

### 3.1 Solution Space of $y'' + y = 0$

Consider the second-order homogeneous ODE

$$y'' + y = 0.$$

Two particular solutions are $y_1(t) = \sin(t)$ and $y_2(t) = \cos(t)$, since

$$\frac{d^2}{dt^2}\sin(t) = -\sin(t) \quad\Longrightarrow\quad -\sin(t) + \sin(t) = 0, \qquad \checkmark$$
$$\frac{d^2}{dt^2}\cos(t) = -\cos(t) \quad\Longrightarrow\quad -\cos(t) + \cos(t) = 0. \qquad \checkmark$$

The **general solution** is

$$y(t) = c_1 \sin(t) + c_2 \cos(t),$$

where $c_1, c_2 \in \mathbb{R}$. This is exactly a **linear combination** of
$\{\sin(t), \cos(t)\}$, so the solution set is the **span** of these two functions.

Since a second-order linear ODE has a 2-dimensional solution space
(Larson, Theorem 4.17), $\{\sin(t), \cos(t)\}$ forms a **basis** if and only if
the two functions are linearly independent.

Let us verify numerically that $\sin(t)$ and $\cos(t)$ are indeed solutions,
and that every linear combination is also a solution.

In [ ]:
def verify_ode_y_double_prime_plus_y(y_func, y_double_prime_func, label):
    """Check y'' + y = 0 at several sample points."""
    test_points = [0.0, 0.5, 1.0, math.pi / 4, math.pi / 2, math.pi, 2.0]
    print(f"Verifying {label}:")
    all_ok = True
    for t in test_points:
        residual = y_double_prime_func(t) + y_func(t)
        ok = abs(residual) < EPSILON
        all_ok = all_ok and ok
        print(f"  t = {t:6.3f}  |  y''(t) + y(t) = {residual:+.2e}  {'OK' if ok else 'FAIL'}")
    print(f"  Result: {'PASSED' if all_ok else 'FAILED'}\n")
    return all_ok


verify_ode_y_double_prime_plus_y(math.sin, lambda t: -math.sin(t), "y = sin(t)")

verify_ode_y_double_prime_plus_y(math.cos, lambda t: -math.cos(t), "y = cos(t)")

c1, c2 = 3.0, -2.0
y_combo = lambda t: c1 * math.sin(t) + c2 * math.cos(t)
y_combo_pp = lambda t: -c1 * math.sin(t) - c2 * math.cos(t)
verify_ode_y_double_prime_plus_y(
    y_combo, y_combo_pp,
    f"y = {c1}*sin(t) + ({c2})*cos(t)  (general solution)"
)

print("The solution space of y'' + y = 0 is spanned by {sin(t), cos(t)}.")
print("It is a 2-dimensional vector space of functions.")

### 3.2 The Wronskian

To confirm that $\{f_1, f_2\}$ are linearly independent as *functions*, we
cannot just row-reduce — they are not column vectors. Instead, we use the
**Wronskian** (Larson, Definition 4.11).

For two functions $f_1, f_2$ that are differentiable on an interval $I$:

$$W(f_1, f_2)(t) = \det \begin{bmatrix} f_1(t) & f_2(t) \\ f_1'(t) & f_2'(t) \end{bmatrix} = f_1(t)\,f_2'(t) - f_2(t)\,f_1'(t).$$

**Theorem (Larson, Theorem 4.18):** If $W(f_1, f_2)(t_0) \neq 0$ for *some*
$t_0 \in I$, then $f_1$ and $f_2$ are linearly independent on $I$.

The Wronskian generalises to $n$ functions $f_1, \ldots, f_n$:

$$W(f_1, \ldots, f_n)(t) = \det \begin{bmatrix}
f_1(t) & f_2(t) & \cdots & f_n(t) \\
f_1'(t) & f_2'(t) & \cdots & f_n'(t) \\
\vdots & \vdots & \ddots & \vdots \\
f_1^{(n-1)}(t) & f_2^{(n-1)}(t) & \cdots & f_n^{(n-1)}(t)
\end{bmatrix}.$$

Notice: this is exactly a **matrix determinant**. The rows are successive
derivatives, and the columns correspond to each function. The entire theory
of Chapter 3 applies.

Let us implement this numerically.

In [ ]:
def wronskian(functions, derivatives, t):
    """Compute the Wronskian of n functions at a point t.

    Parameters
    ----------
    functions : list of callables
        [f1, f2, ..., fn] — the original functions.
    derivatives : list of lists of callables
        derivatives[k] = [f1^(k+1), f2^(k+1), ..., fn^(k+1)]
        So derivatives[0] holds the first derivatives,
           derivatives[1] holds the second derivatives, etc.
        Must have len(derivatives) == len(functions) - 1.
    t : float
        The point at which to evaluate.

    Returns
    -------
    float
        The Wronskian W(f1, ..., fn)(t).
    """
    n = len(functions)
    if len(derivatives) != n - 1:
        raise ValueError(
            f"Need {n - 1} rows of derivatives for {n} functions, "
            f"got {len(derivatives)}"
        )

    rows = []
    rows.append([f(t) for f in functions])
    for deriv_row in derivatives:
        rows.append([d(t) for d in deriv_row])

    W = Matrix.from_lists(rows)
    det_val = det_cofactor(W)
    return det_val


print("wronskian() defined.")
print("  Signature: wronskian(functions, derivatives, t) -> float")
print("  Uses det_cofactor from linalg_core to compute the determinant.")

### 3.3 Demo — Independence of $\sin(t)$ and $\cos(t)$

We evaluate the Wronskian of $f_1 = \sin(t)$ and $f_2 = \cos(t)$ at $t = 0$.

**Step-by-step:**

$$f_1(0) = \sin(0) = 0, \qquad f_2(0) = \cos(0) = 1,$$
$$f_1'(0) = \cos(0) = 1, \qquad f_2'(0) = -\sin(0) = 0.$$

So the Wronskian matrix at $t = 0$ is

$$\begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix}$$

and its determinant is

$$W = (0)(0) - (1)(1) = -1 \neq 0.$$

Since $W \neq 0$, the functions $\sin(t)$ and $\cos(t)$ are **linearly independent**.
Therefore $\{\sin(t), \cos(t)\}$ is a **basis** for the solution space of $y'' + y = 0$.

In [ ]:
f1 = math.sin
f2 = math.cos

f1_prime = math.cos
f2_prime = lambda t: -math.sin(t)

t0 = 0.0

print("Functions: f1 = sin(t), f2 = cos(t)")
print(f"Evaluating at t = {t0}:\n")
print(f"  f1({t0}) = sin({t0}) = {f1(t0):.4f}")
print(f"  f2({t0}) = cos({t0}) = {f2(t0):.4f}")
print(f"  f1'({t0}) = cos({t0}) = {f1_prime(t0):.4f}")
print(f"  f2'({t0}) = -sin({t0}) = {f2_prime(t0):.4f}")

W_matrix = Matrix.from_lists([
    [f1(t0), f2(t0)],
    [f1_prime(t0), f2_prime(t0)]
])
print(f"\nWronskian matrix at t = {t0}:")
print(W_matrix)

W_val = wronskian([f1, f2], [[f1_prime, f2_prime]], t0)
print(f"\nW(sin, cos)(0) = {W_val:.4f}")
print(f"W != 0? {abs(W_val) > EPSILON}")
print("\nConclusion: sin(t) and cos(t) are LINEARLY INDEPENDENT.")
print("{sin(t), cos(t)} is a basis for the solution space of y'' + y = 0.")

print("\n--- Wronskian at several other points ---")
for t in [0.0, 0.5, 1.0, math.pi / 4, math.pi / 2, math.pi]:
    w = wronskian([f1, f2], [[f1_prime, f2_prime]], t)
    print(f"  W(sin, cos)({t:6.3f}) = {w:+.6f}")
print("\nNote: W = -1 everywhere (constant). This is a property of")
print("solutions to y'' + y = 0 (Abel's identity).")

### 3.4 Higher Order — $y''' - y = 0$ (3×3 Wronskian)

The ODE $y''' - y = 0$ has the characteristic equation $r^3 - 1 = 0$, whose
real root is $r = 1$. The full factorisation is $(r - 1)(r^2 + r + 1) = 0$,
giving complex roots as well. Three real-valued fundamental solutions are:

$$y_1(t) = e^t, \qquad y_2(t) = e^{-t/2}\cos\!\left(\frac{\sqrt{3}}{2}\,t\right), \qquad y_3(t) = e^{-t/2}\sin\!\left(\frac{\sqrt{3}}{2}\,t\right).$$

For a third-order ODE, we need a $3 \times 3$ Wronskian:

$$W(y_1, y_2, y_3)(t) = \det \begin{bmatrix}
y_1 & y_2 & y_3 \\
y_1' & y_2' & y_3' \\
y_1'' & y_2'' & y_3''
\end{bmatrix}.$$

If $W \neq 0$ at any point, the three functions are linearly independent and
form a basis for the 3-dimensional solution space.

In [ ]:
sqrt3_over_2 = math.sqrt(3) / 2

y1 = lambda t: math.exp(t)
y2 = lambda t: math.exp(-t / 2) * math.cos(sqrt3_over_2 * t)
y3 = lambda t: math.exp(-t / 2) * math.sin(sqrt3_over_2 * t)

y1_p = lambda t: math.exp(t)
y2_p = lambda t: (-0.5 * math.exp(-t / 2) * math.cos(sqrt3_over_2 * t)
                  - sqrt3_over_2 * math.exp(-t / 2) * math.sin(sqrt3_over_2 * t))
y3_p = lambda t: (-0.5 * math.exp(-t / 2) * math.sin(sqrt3_over_2 * t)
                  + sqrt3_over_2 * math.exp(-t / 2) * math.cos(sqrt3_over_2 * t))

y1_pp = lambda t: math.exp(t)
y2_pp = lambda t: (0.25 * math.exp(-t / 2) * math.cos(sqrt3_over_2 * t)
                   + 0.5 * sqrt3_over_2 * math.exp(-t / 2) * math.sin(sqrt3_over_2 * t)
                   + 0.5 * sqrt3_over_2 * math.exp(-t / 2) * math.sin(sqrt3_over_2 * t)
                   - (sqrt3_over_2 ** 2) * math.exp(-t / 2) * math.cos(sqrt3_over_2 * t))
y3_pp = lambda t: (0.25 * math.exp(-t / 2) * math.sin(sqrt3_over_2 * t)
                   - 0.5 * sqrt3_over_2 * math.exp(-t / 2) * math.cos(sqrt3_over_2 * t)
                   - 0.5 * sqrt3_over_2 * math.exp(-t / 2) * math.cos(sqrt3_over_2 * t)
                   - (sqrt3_over_2 ** 2) * math.exp(-t / 2) * math.sin(sqrt3_over_2 * t))

print("Solutions of y''' - y = 0:")
print("  y1(t) = e^t")
print("  y2(t) = e^(-t/2) * cos(sqrt(3)/2 * t)")
print("  y3(t) = e^(-t/2) * sin(sqrt(3)/2 * t)")

t0 = 0.0
print(f"\n3x3 Wronskian matrix at t = {t0}:")
W_mat = Matrix.from_lists([
    [y1(t0), y2(t0), y3(t0)],
    [y1_p(t0), y2_p(t0), y3_p(t0)],
    [y1_pp(t0), y2_pp(t0), y3_pp(t0)]
])
print(W_mat)

W3 = wronskian(
    [y1, y2, y3],
    [[y1_p, y2_p, y3_p],
     [y1_pp, y2_pp, y3_pp]],
    t0
)
print(f"\nW(y1, y2, y3)(0) = {W3:.6f}")
print(f"W != 0? {abs(W3) > EPSILON}")

print("\nConclusion: y1, y2, y3 are LINEARLY INDEPENDENT.")
print("They form a basis for the 3-dimensional solution space of y''' - y = 0.")

print("\n--- Verify y1 = e^t satisfies y''' - y = 0 ---")
for t in [0.0, 0.5, 1.0, 2.0]:
    residual = y1_pp(t) - y1(t)  # y1''' = e^t, but y1''' = y1_pp' = e^t
    print(f"  t = {t:.1f}: y'''(t) - y(t) = e^t - e^t = {math.exp(t) - math.exp(t):.2e}  OK")

### 3.5 Connection to Matrices

The Wronskian is *just a determinant*. Every theorem from Chapter 3 applies:

| Chapter 3 concept | Wronskian interpretation |
|---|---|
| $\det(A) = 0$ iff columns are linearly dependent | $W = 0$ iff functions are dependent |
| $\det(A) \neq 0$ iff $A$ is invertible | $W \neq 0$ iff functions are independent |
| Row/column operations simplify $\det$ | Derivative identities simplify $W$ |
| $\det(AB) = \det(A)\det(B)$ | Composition rules for Wronskians |

The vector space here is $C^\infty(I)$, the space of infinitely differentiable
functions on an interval $I$, rather than $\mathbb{R}^n$. But the
**abstract structure is identical**:

- **Vectors** $\rightarrow$ functions.
- **Linear combination** $\rightarrow$ $c_1 f_1 + c_2 f_2 + \cdots$.
- **Basis** $\rightarrow$ fundamental set of solutions.
- **Dimension** $\rightarrow$ order of the ODE.
- **Determinant test for independence** $\rightarrow$ Wronskian.

This is the power of abstraction: one theory, many vector spaces.

Let us make this explicit by showing that the Wronskian matrix is a perfectly
ordinary `Matrix` object, and `det_cofactor` computes it the same way as any
other determinant.

In [ ]:
print("=" * 60)
print("THE WRONSKIAN IS JUST A DETERMINANT")
print("=" * 60)

print("\n--- Example 1: sin(t), cos(t) at t = 0 ---")
W_22 = Matrix.from_lists([
    [math.sin(0), math.cos(0)],
    [math.cos(0), -math.sin(0)]
])
print("Wronskian matrix (a regular 2x2 Matrix object):")
print(W_22)
print(f"det_cofactor:     {det_cofactor(W_22):.4f}")
print(f"det_elimination:  {det_elimination(W_22):.4f}")
print(f"wronskian():      {wronskian([math.sin, math.cos], [[math.cos, lambda t: -math.sin(t)]], 0):.4f}")
print("All three agree: same computation, same answer.")

print("\n--- Example 2: e^t, e^(-t/2)cos(...), e^(-t/2)sin(...) at t = 0 ---")
W_33 = Matrix.from_lists([
    [y1(0), y2(0), y3(0)],
    [y1_p(0), y2_p(0), y3_p(0)],
    [y1_pp(0), y2_pp(0), y3_pp(0)]
])
print("Wronskian matrix (a regular 3x3 Matrix object):")
print(W_33)
print(f"det_cofactor:     {det_cofactor(W_33):.6f}")
print(f"det_elimination:  {det_elimination(W_33):.6f}")

print("\n" + "=" * 60)
print("TAKEAWAY")
print("=" * 60)
print("The Wronskian is not a new concept. It is the determinant of")
print("Chapter 3 applied to a new vector space (function spaces).")
print("Same linear algebra, different vectors.")

---

## 4. Exercises

Test your understanding with the following problems. Write code in the provided cells.

### Exercise 1 (Easy) — Wronskian of $t$ and $t^2$

Compute the Wronskian of $f_1(t) = t$ and $f_2(t) = t^2$ at $t = 1$.

**Recall:**
- $f_1'(t) = 1$
- $f_2'(t) = 2t$

So
$$W(t, t^2)(t) = \det \begin{bmatrix} t & t^2 \\ 1 & 2t \end{bmatrix} = 2t^2 - t^2 = t^2.$$

Use the `wronskian()` function to verify that $W(1) = 1$.
Then evaluate at $t = 0$ and explain what happens.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium) — Independence of $e^t$, $e^{2t}$, $e^{3t}$

Show that $f_1(t) = e^t$, $f_2(t) = e^{2t}$, and $f_3(t) = e^{3t}$ are linearly
independent by computing the $3 \times 3$ Wronskian.

**Derivatives:**

| | $f_1 = e^t$ | $f_2 = e^{2t}$ | $f_3 = e^{3t}$ |
|---|---|---|---|
| $f$ | $e^t$ | $e^{2t}$ | $e^{3t}$ |
| $f'$ | $e^t$ | $2e^{2t}$ | $3e^{3t}$ |
| $f''$ | $e^t$ | $4e^{2t}$ | $9e^{3t}$ |

Evaluate at $t = 0$. The Wronskian matrix should be a Vandermonde-like matrix.
Compute its determinant and verify it is nonzero.

*Hint:* At $t = 0$ every exponential equals 1, so the matrix simplifies to
$\begin{bmatrix} 1 & 1 & 1 \\ 1 & 2 & 3 \\ 1 & 4 & 9 \end{bmatrix}$.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge) — Basis for $y'' - 3y' + 2y = 0$

The ODE $y'' - 3y' + 2y = 0$ has the characteristic equation $r^2 - 3r + 2 = 0$,
which factors as $(r - 1)(r - 2) = 0$. So the two fundamental solutions are

$$y_1(t) = e^t, \qquad y_2(t) = e^{2t}.$$

**Part A:** Verify numerically that $y_1$ and $y_2$ each satisfy the ODE. Check
that $y'' - 3y' + 2y = 0$ at several values of $t$.

**Part B:** Compute the Wronskian $W(e^t, e^{2t})(t)$ at $t = 0$ and confirm
it is nonzero, proving the two solutions are linearly independent.

**Part C:** Verify that a general linear combination $y = c_1 e^t + c_2 e^{2t}$
also satisfies the ODE (pick specific values of $c_1, c_2$).

*Derivatives:*
- $y_1 = e^t$, $y_1' = e^t$, $y_1'' = e^t$.
- $y_2 = e^{2t}$, $y_2' = 2e^{2t}$, $y_2'' = 4e^{2t}$.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | $\mathbb{R}^n$ (Chapters 1–3) | Function spaces (Section 4.8) |
|---|---|---|
| **Vector** | Column vector $\mathbf{v} \in \mathbb{R}^n$ | Function $f(t)$ on interval $I$ |
| **Linear combination** | $c_1 \mathbf{v}_1 + \cdots + c_n \mathbf{v}_n$ | $c_1 f_1(t) + \cdots + c_n f_n(t)$ |
| **Independence test** | $\det(A) \neq 0$ (columns of $A$) | $W(f_1, \ldots, f_n)(t_0) \neq 0$ |
| **Basis** | $n$ independent columns | $n$ fundamental solutions |
| **Dimension** | Number of columns in basis | Order of the ODE |
| **Key computation** | Determinant (Ch. 3) | Wronskian = determinant of derivatives |

**Takeaway:** The theory of vector spaces is not limited to columns of numbers.
Differential equations have solution spaces that are genuine vector spaces, and
the Wronskian — which is simply a determinant — tests whether solutions are
independent. The abstraction of Chapter 4 unifies these seemingly different worlds
under one algebraic framework.